# Two-Stage EfficientNet-B0 Fine-Tuning Pipeline for Deepfake Detection
### Google Colab GPU Edition

This notebook trains a binary **EfficientNet-B0** deepfake detector on a high-resolution Kaggle Real vs Fake face dataset (e.g., `xhlulu/140k-real-and-fake-faces`).

**Pipeline Highlights:**
1. **Kaggle API Integration**: Securely downloads and unzips dataset.
2. **Data Cleaning & Deduplication**: Discards corrupt images and checks MD5/dHash duplicates.
3. **Leakage-Free Splitting**: 70% Train, 15% Validation, 15% Test with identity grouping.
4. **Conservative Augmentation**: Flips, subtle rotations, and mild jitter that preserve subtle generative artifacts.
5. **Two-Stage Fine-Tuning**:
   - Stage 1: Freeze backbone, train classification head (`AdamW`, lr=1e-3, 4 epochs).
   - Stage 2: Unfreeze top blocks 6 & 7, fine-tune (`AdamW`, lr=1e-5, CosineAnnealing, 8 epochs).
6. **Post-Training Temperature Scaling**: Calibrates softmax probabilities.
7. **Test Evaluation**: Confusion Matrix, ROC-AUC, Fake-Recall, Precision, F1-Score.
8. **Grad-CAM Visualization**: Saliency heatmaps indicating spatial feature attribution.

## Step 1: GPU Verification & Dependency Installation

In [ ]:
# Verify GPU availability in Colab
!nvidia-smi

# Install dependencies
!pip install -q kaggle scikit-learn seaborn matplotlib tqdm pillow opencv-python-headless

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## Step 2: Configure Kaggle API Credentials

1. Go to your Kaggle Account page (`https://www.kaggle.com/settings`).
2. Scroll down to the **API** section and click **Create New Token** to download `kaggle.json`.
3. Run the cell below to upload `kaggle.json`.

In [ ]:
from google.colab import files
import os
import shutil

print("Please upload your kaggle.json file:")
uploaded = files.upload()

os.makedirs("/root/.kaggle", exist_ok=True)
for fn in uploaded.keys():
    shutil.move(fn, "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("Kaggle API token configured successfully!")

## Step 3: Download and Extract Kaggle Real vs Fake Dataset

We recommend `xhlulu/140k-real-and-fake-faces` (high-resolution human face dataset: Flickr-Faces-HQ authentic portraits vs StyleGAN synthetic faces).

In [ ]:
DATASET_SLUG = "xhlulu/140k-real-and-fake-faces"
RAW_DATA_DIR = "/content/raw_dataset"

!kaggle datasets download -d {DATASET_SLUG} -p {RAW_DATA_DIR} --unzip
print("Dataset downloaded and extracted!")
!ls -lah {RAW_DATA_DIR}

## Step 4: Clone / Download Pipeline Scripts or Run Directly

In [ ]:
# Create project directory structure
!mkdir -p deepfake_detection/src deepfake_detection/models deepfake_detection/results deepfake_detection/dataset

print("Environment initialized!")

## Step 5: Data Inspection, Cleaning, & Leakage-Free Splitting

In [ ]:
import sys
sys.path.append("deepfake_detection")

# If using 140k-real-and-fake-faces, it already provides train/val/test splits!
# Check directory structure
import os
contents = os.listdir(RAW_DATA_DIR)
print("Dataset contents:", contents)

# Check if train/valid/test exist inside
split_folders = [f for f in os.listdir(RAW_DATA_DIR) if os.path.isdir(os.path.join(RAW_DATA_DIR, f))]
print("Split folders found:", split_folders)

## Step 6: Execute Two-Stage Fine-Tuning on Colab GPU

In [ ]:
# Execute the master pipeline
# Adjust batch-size to 64 if running on A100 GPU, or 32 on T4 GPU
!python deepfake_detection/run_pipeline.py \
    --raw-data "/content/raw_dataset" \
    --data-dir "deepfake_detection/dataset" \
    --stage1-epochs 4 \
    --stage2-epochs 8 \
    --batch-size 32 \
    --stage1-lr 0.001 \
    --stage2-lr 0.00001 \
    --patience 4

## Step 7: View Training Curves & Evaluation Visualizations

In [ ]:
from IPython.display import Image, display

print("=== Training Loss ===")
display(Image("deepfake_detection/results/training_loss.png"))

print("=== Training Accuracy ===")
display(Image("deepfake_detection/results/training_accuracy.png"))

print("=== Confusion Matrix ===")
display(Image("deepfake_detection/results/confusion_matrix.png"))

print("=== ROC Curve ===")
display(Image("deepfake_detection/results/roc_curve.png"))

## Step 8: Single-Image Inference & Grad-CAM Explainability

In [ ]:
from deepfake_detection.src.predict import predict_image
import glob

# Find a test image to evaluate
test_images = glob.glob("deepfake_detection/dataset/test/*/*.*")
if test_images:
    sample_img = test_images[0]
    print(f"Evaluating sample image: {sample_img}")
    res = predict_image(
        image_path=sample_img,
        model_path="deepfake_detection/models/best_model.pth",
        generate_gradcam=True,
    )
    if res.get("gradcam_overlay_path"):
        display(Image(res["gradcam_overlay_path"]))

## Step 9: Download Trained Checkpoint to Local Machine

Download `best_model.pth` and place it in your local project directory at `explainable_deepfake_detection/models/classifier_deepfake_weights.pt` to integrate directly into your Classifier Agent!

In [ ]:
from google.colab import files
files.download("deepfake_detection/models/best_model.pth")